In [1]:
import os
import re
import json
import torch
import numpy as np
import pandas as pd

from scipy.special import softmax
from transformers import AutoTokenizer, AutoModelForSequenceClassification

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

SARCASM_MODEL_DIR = "/kaggle/input/models/kunjpatel1607/sarcasm-v6/transformers/default/1"
EMOTION_MODEL_DIR = "/kaggle/input/models/kunjpatel1607/emotion-v2-production/transformers/default/1"

SARCASM_THRESHOLD = 0.34
MAX_LENGTH = 128

print("Device:", DEVICE)
print("Sarcasm model:", SARCASM_MODEL_DIR)
print("Emotion model:", EMOTION_MODEL_DIR)

Device: cuda
Sarcasm model: /kaggle/input/models/kunjpatel1607/sarcasm-v6/transformers/default/1
Emotion model: /kaggle/input/models/kunjpatel1607/emotion-v2-production/transformers/default/1


In [2]:
def check_model_dir(model_dir, name):
    required_files = [
        "config.json",
        "model.safetensors",
        "tokenizer.json",
        "tokenizer_config.json",
    ]

    print(f"\n{name} model check:")

    for file in required_files:
        path = os.path.join(model_dir, file)
        print(file, "✅" if os.path.exists(path) else "❌")


check_model_dir(SARCASM_MODEL_DIR, "Sarcasm")
check_model_dir(EMOTION_MODEL_DIR, "Emotion")


Sarcasm model check:
config.json ✅
model.safetensors ✅
tokenizer.json ✅
tokenizer_config.json ✅

Emotion model check:
config.json ✅
model.safetensors ✅
tokenizer.json ✅
tokenizer_config.json ✅


In [3]:
sarcasm_tokenizer = AutoTokenizer.from_pretrained(SARCASM_MODEL_DIR)
sarcasm_model = AutoModelForSequenceClassification.from_pretrained(
    SARCASM_MODEL_DIR
).to(DEVICE)
sarcasm_model.eval()

emotion_tokenizer = AutoTokenizer.from_pretrained(EMOTION_MODEL_DIR)
emotion_model = AutoModelForSequenceClassification.from_pretrained(
    EMOTION_MODEL_DIR
).to(DEVICE)
emotion_model.eval()

print("Sarcasm labels:", sarcasm_model.config.id2label)
print("Emotion labels:", emotion_model.config.id2label)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Sarcasm labels: {0: 'Not Sarcastic', 1: 'Sarcastic'}
Emotion labels: {0: 'anger', 1: 'annoyance', 2: 'confusion', 3: 'disappointment', 4: 'fear', 5: 'gratitude', 6: 'joy', 7: 'love', 8: 'neutral', 9: 'optimism', 10: 'sadness', 11: 'surprise'}


In [4]:
def split_into_statements(text):
    text = str(text).strip()

    if not text:
        return []

    text = re.sub(r"\r\n|\r", "\n", text)

    lines = [
        line.strip()
        for line in text.split("\n")
        if line.strip()
    ]

    statements = []

    for line in lines:
        parts = re.split(r"(?<=[.!?])\s+", line)

        for part in parts:
            part = part.strip()

            if not part:
                continue

            part = re.sub(r"\.{2,}", ".", part)
            part = re.sub(r"\!{2,}", "!", part)
            part = re.sub(r"\?{2,}", "?", part)

            statements.append(part)

    return statements

In [5]:
def predict_sarcasm(text):
    inputs = sarcasm_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
    ).to(DEVICE)

    with torch.no_grad():
        outputs = sarcasm_model(**inputs)

    probs = softmax(
        outputs.logits.detach().cpu().numpy(),
        axis=1,
    )[0]

    sarcasm_prob = float(probs[1])

    label = (
        "Sarcastic"
        if sarcasm_prob >= SARCASM_THRESHOLD
        else "Not Sarcastic"
    )

    return {
        "sarcasm_label": label,
        "sarcasm_score": round(sarcasm_prob * 100, 2),
        "sarcasm_probability": sarcasm_prob,
        "sarcasm_reason": "Raw model probability.",
    }


def predict_emotion(text):
    inputs = emotion_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
    ).to(DEVICE)

    with torch.no_grad():
        outputs = emotion_model(**inputs)

    probs = softmax(
        outputs.logits.detach().cpu().numpy(),
        axis=1,
    )[0]

    top_idx = int(np.argmax(probs))
    top_label = emotion_model.config.id2label[top_idx]
    top_score = float(probs[top_idx])

    top3_idx = probs.argsort()[-3:][::-1]

    top3 = [
        {
            "emotion": emotion_model.config.id2label[int(i)],
            "score": round(float(probs[int(i)]) * 100, 2),
        }
        for i in top3_idx
    ]

    return {
        "primary_emotion": top_label,
        "emotion_score": round(top_score * 100, 2),
        "emotion_probability": top_score,
        "top3_emotions": top3,
    }

In [6]:
NEGATIVE_EMOTIONS = {
    "anger",
    "annoyance",
    "sadness",
    "fear",
    "disappointment",
}

POSITIVE_EMOTIONS = {
    "joy",
    "love",
    "gratitude",
    "optimism",
}

NEUTRAL_EMOTIONS = {
    "neutral",
    "confusion",
    "surprise",
}

In [21]:
POSITIVE_WORD_CUES = [
    "great",
    "amazing",
    "wonderful",
    "perfect",
    "fantastic",
    "excellent",
    "brilliant",
    "awesome",
    "lovely",
    "beautiful",
    "nice",
    "fun",
    "helpful",
    "convenient",
    "incredible",
    "impressive",
    "smooth",
    "peaceful",
    "calm",
    "happy",
    "glad",
    "proud",
    "thankful",
    "grateful",
    "appreciate",
    "love",
    "enjoy",
    "enjoying",
    "relieved",
    "satisfied",
    "confident",
    "motivated",
]

NEGATIVE_EVENT_CUES = [
    "crash",
    "crashed",
    "crashing",
    "fail",
    "failed",
    "failure",
    "rejected",
    "reject",
    "ignored",
    "ignore",
    "ruined",
    "ruin",
    "broke",
    "broken",
    "corrupt",
    "corrupted",
    "deleted",
    "lost",
    "error",
    "bug",
    "issue",
    "problem",
    "wrong",
    "worse",
    "worst",
    "missed",
    "late",
    "deadline",
    "pressure",
    "stress",
    "stressed",
    "anxious",
    "anxiety",
    "overwhelmed",
    "exhausted",
    "tired",
    "drained",
    "burnt out",
    "burned out",
    "lonely",
    "sad",
    "upset",
    "hurt",
    "disappointed",
    "angry",
    "annoyed",
    "frustrated",
    "confused",
    "worried",
    "scared",
    "nervous",
    "blamed",
    "blame",
    "cancelled",
    "canceled",
    "cancel",
    "betrayed",
    "betray",
    "lied",
    "lied to",
    "disrespected",
    "insulted",
    "humiliated",
    "excluded",
    "left out",
    "treated badly",
    "treated poorly",
    "treated unfairly",
    "not respected",
    "not valued",
    "made fun of",
]

CONTRAST_CUES = [
    "but",
    "except",
    "although",
    "though",
    "however",
    "yet",
    "still",
    "instead",
    "even though",
    "only to",
]

INTENSIFIER_CUES = [
    "so",
    "really",
    "absolutely",
    "totally",
    "definitely",
    "literally",
    "obviously",
    "clearly",
    "exactly",
    "just",
]

SARCASM_PATTERN_CUES = [
    "what a",
    "how nice",
    "how convenient",
    "just what",
    "exactly what",
    "couldn't have asked",
    "love when",
    "i love when",
    "i just love",
    "thanks for",
    "thank you for",
    "thanks a lot",
    "very kind",
    "very helpful",
    "as if",
    "because apparently",
    "really appreciate",
    "appreciate being",
]

VAGUE_UNCERTAIN_CUES = [
    "something",
    "interesting",
    "one way",
    "i guess",
    "apparently",
    "sure",
    "fine",
    "okay then",
    "unexpected",
    "well",
]

POSITIVE_RESOLUTION_CUES = [
    "solved",
    "fixed",
    "resolved",
    "completed",
    "finished",
    "improved",
    "finally works",
    "works now",
    "went well",
    "successfully",
    "relieved",
    "satisfied",
    "proud",
    "confident",
    "prepared",
    "motivated",
]

NEUTRAL_ACTIVITY_CUES = [
    "went to",
    "attended",
    "lecture",
    "lectures",
    "meeting started",
    "meeting ended",
    "submitted",
    "received",
    "confirmation",
    "cleaned",
    "arranged",
    "had lunch",
    "studied",
    "filled",
    "opened",
    "closed",
    "checked",
    "updated",
    "wrote down",
    "read",
    "walked",
]

SHORT_AMBIGUOUS_POSITIVE = [
    "nice",
    "sure",
    "fine",
    "okay",
    "cool",
    "great",
    "amazing",
    "perfect",
]

# =========================================================
# Final V1 patch vocabulary
# General category-level cues, not sentence-specific
# =========================================================

SUPPORTIVE_POSITIVE_CUES = [
    "supported",
    "support",
    "helped",
    "help",
    "encouraged",
    "encourage",
    "comforted",
    "understood",
    "listened",
    "guided",
    "motivated",
    "stood by",
    "was there for me",
    "received support",
]

STUDENT_NEGATIVE_CUES = [
    "surprise test",
    "test",
    "exam",
    "assignment",
    "submission",
    "deadline",
    "harder",
    "difficult paper",
    "unexpected question",
    "too many deadlines",
    "unfinished assignment",
    "before i finished",
    "before finishing",
]

CODING_NEGATIVE_CUES = [
    "debugging",
    "works locally",
    "fails online",
    "deployment",
    "backend",
    "frontend",
    "api",
    "server",
    "build failed",
    "runtime error",
    "same code",
]

CLEAR_POSITIVE_STUDY_CUES = [
    "studied well",
    "prepared",
    "feel prepared",
    "completed my notes",
    "finished my notes",
    "revised well",
    "ready for the exam",
    "confident for the exam",
]

EXPECTED_NEUTRAL_UNCERTAIN_CUES = [
    "exactly what i expected",
    "what i expected",
    "as expected",
]

In [23]:
def text_lower(text):
    return str(text).lower()


def has_any(text, cues):
    t = text_lower(text)
    return any(cue in t for cue in cues)


def cue_count(text, cues):
    t = text_lower(text)
    return sum(1 for cue in cues if cue in t)


def has_positive_surface(text):
    return has_any(text, POSITIVE_WORD_CUES)


def has_negative_event(text):
    return has_any(text, NEGATIVE_EVENT_CUES)


def has_contrast(text):
    return has_any(text, CONTRAST_CUES)


def has_sarcasm_pattern(text):
    return has_any(text, SARCASM_PATTERN_CUES)


def has_vague_uncertain_tone(text):
    return has_any(text, VAGUE_UNCERTAIN_CUES)


def has_positive_resolution(text):
    return has_any(text, POSITIVE_RESOLUTION_CUES)


def has_neutral_activity(text):
    return has_any(text, NEUTRAL_ACTIVITY_CUES)


def has_exhaustion_or_stress(text):
    return has_any(
        text,
        [
            "exhausted",
            "overwhelmed",
            "tired",
            "drained",
            "burnt out",
            "burned out",
            "stressed",
            "anxious",
            "pressure",
            "mentally tired",
            "emotionally tired",
        ],
    )


def is_short_ambiguous_positive(text):
    t = text_lower(text).strip()
    t = re.sub(r"[.!?]+$", "", t).strip()

    word_count = len(t.split())

    return (
        word_count <= 4
        and any(
            t == cue or t.startswith(cue + ",")
            for cue in SHORT_AMBIGUOUS_POSITIVE
        )
    )


def is_literal_positive_resolution(text):
    return (
        has_positive_resolution(text)
        and has_positive_surface(text)
    )


def is_literal_neutral_activity(text, emotion_result=None):
    emotion_score = 0

    if emotion_result is not None:
        emotion_score = emotion_result.get("emotion_score", 0)

    return (
        has_neutral_activity(text)
        and not has_negative_event(text)
        and not has_positive_resolution(text)
        and not has_sarcasm_pattern(text)
        and emotion_score < 55
    )


def is_literal_negative_statement(text, emotion):
    return (
        has_negative_event(text)
        and not has_positive_surface(text)
        and not has_sarcasm_pattern(text)
        and not has_contrast(text)
        and emotion in NEGATIVE_EMOTIONS
    )


def contradiction_strength(text, previous_negative_context=False):
    """
    General contradiction score:
    positive wording + negative event/context = sarcasm probability boost.
    Positive recovery/support/study-preparation should not become sarcasm.
    """

    # Genuine positive resolution/support/study preparation should not become sarcasm.
    if is_literal_positive_resolution(text):
        return 0

    score = 0

    if has_positive_surface(text):
        score += 1

    if has_negative_event(text):
        score += 1

    if has_sarcasm_pattern(text):
        score += 1

    if has_contrast(text):
        score += 1

    if previous_negative_context and has_positive_surface(text):
        score += 1

    if (
        cue_count(text, INTENSIFIER_CUES) >= 1
        and has_positive_surface(text)
        and has_negative_event(text)
    ):
        score += 1

    # Student/coding negative with positive surface is often sarcasm.
    if (
        has_positive_surface(text)
        and (
            has_student_negative_event(text)
            or has_coding_negative_context(text)
        )
        and not is_literal_positive_resolution(text)
    ):
        score += 1

    return score

# =========================================================
# Final V1 patch helper functions
# These override/extend earlier helpers
# =========================================================

def has_supportive_positive(text):
    return has_any(text, SUPPORTIVE_POSITIVE_CUES)


def has_student_negative_event(text):
    return has_any(text, STUDENT_NEGATIVE_CUES)


def has_coding_negative_context(text):
    return has_any(text, CODING_NEGATIVE_CUES)


def has_clear_positive_study(text):
    return has_any(text, CLEAR_POSITIVE_STUDY_CUES)


def has_expected_uncertain_tone(text):
    return has_any(text, EXPECTED_NEUTRAL_UNCERTAIN_CUES)


# Override earlier version
def has_negative_event(text):
    return (
        has_any(text, NEGATIVE_EVENT_CUES)
        or has_student_negative_event(text)
        or has_coding_negative_context(text)
    )


# Override earlier version
def is_literal_positive_resolution(text):
    return (
        has_positive_resolution(text)
        or has_supportive_positive(text)
        or has_clear_positive_study(text)
    )

In [24]:
def calibrate_sarcasm(text, emotion_result, sarcasm_result, previous_negative_context=False):
    emotion = emotion_result["primary_emotion"]
    sarcasm_prob = sarcasm_result["sarcasm_probability"]

    contradiction = contradiction_strength(
        text,
        previous_negative_context=previous_negative_context,
    )

    # 1. Genuine positive resolution/completion
    if is_literal_positive_resolution(text):
        sarcasm_result["sarcasm_probability"] = min(sarcasm_prob, 0.20)
        sarcasm_result["sarcasm_score"] = round(sarcasm_result["sarcasm_probability"] * 100, 2)
        sarcasm_result["sarcasm_label"] = "Not Sarcastic"
        sarcasm_result["sarcasm_reason"] = "Positive resolution/completion statement."
        return sarcasm_result

    # 2. Short ambiguous positive word without negative context
    if (
        is_short_ambiguous_positive(text)
        and not previous_negative_context
        and not has_negative_event(text)
    ):
        sarcasm_result["sarcasm_probability"] = min(sarcasm_prob, 0.30)
        sarcasm_result["sarcasm_score"] = round(sarcasm_result["sarcasm_probability"] * 100, 2)
        sarcasm_result["sarcasm_label"] = "Uncertain"
        sarcasm_result["sarcasm_reason"] = "Short ambiguous positive phrase without context."
        return sarcasm_result

    # 3. Literal neutral daily activity
    if is_literal_neutral_activity(text, emotion_result):
        sarcasm_result["sarcasm_probability"] = min(sarcasm_prob, 0.15)
        sarcasm_result["sarcasm_score"] = round(sarcasm_result["sarcasm_probability"] * 100, 2)
        sarcasm_result["sarcasm_label"] = "Not Sarcastic"
        sarcasm_result["sarcasm_reason"] = "Literal neutral activity."
        return sarcasm_result

    # 4. Literal negative sentence
    if is_literal_negative_statement(text, emotion):
        sarcasm_result["sarcasm_probability"] = min(sarcasm_prob, 0.20)
        sarcasm_result["sarcasm_score"] = round(sarcasm_result["sarcasm_probability"] * 100, 2)
        sarcasm_result["sarcasm_label"] = "Not Sarcastic"
        sarcasm_result["sarcasm_reason"] = "Literal negative statement."
        return sarcasm_result

        # 4.5 Expected/as expected phrases are usually ambiguous unless negative context exists.
    if (
        has_expected_uncertain_tone(text)
        and not has_negative_event(text)
        and not previous_negative_context
    ):
        sarcasm_result["sarcasm_probability"] = max(min(sarcasm_prob, 0.32), 0.30)
        sarcasm_result["sarcasm_score"] = round(
            sarcasm_result["sarcasm_probability"] * 100,
            2,
        )
        sarcasm_result["sarcasm_label"] = "Uncertain"
        sarcasm_result["sarcasm_reason"] = "Expected/uncertain phrase without negative context."
        return sarcasm_result

    # 4.6 Student/coding sarcasm:
    # positive surface + academic/technical negative context.
    if (
        has_positive_surface(text)
        and (
            has_student_negative_event(text)
            or has_coding_negative_context(text)
        )
        and not is_literal_positive_resolution(text)
    ):
        sarcasm_result["sarcasm_probability"] = max(sarcasm_prob, 0.85)
        sarcasm_result["sarcasm_score"] = round(
            sarcasm_result["sarcasm_probability"] * 100,
            2,
        )
        sarcasm_result["sarcasm_label"] = "Sarcastic"
        sarcasm_result["sarcasm_reason"] = (
            "Positive wording conflicts with student/coding negative context."
        )
        return sarcasm_result
    

    # 5. Strong contradiction
    if contradiction >= 3:
        sarcasm_result["sarcasm_probability"] = max(sarcasm_prob, 0.85)
        sarcasm_result["sarcasm_score"] = round(sarcasm_result["sarcasm_probability"] * 100, 2)
        sarcasm_result["sarcasm_label"] = "Sarcastic"
        sarcasm_result["sarcasm_reason"] = "Positive wording conflicts with negative context."
        return sarcasm_result

    # 6. Medium contradiction
    if contradiction == 2 and has_positive_surface(text) and has_negative_event(text):
        sarcasm_result["sarcasm_probability"] = max(sarcasm_prob, 0.75)
        sarcasm_result["sarcasm_score"] = round(sarcasm_result["sarcasm_probability"] * 100, 2)
        sarcasm_result["sarcasm_label"] = "Likely Sarcastic"
        sarcasm_result["sarcasm_reason"] = "Positive wording conflicts with negative event."
        return sarcasm_result

    # 7. Positive wording after previous negative context
    if previous_negative_context and has_positive_surface(text):
        sarcasm_result["sarcasm_probability"] = max(sarcasm_prob, 0.75)
        sarcasm_result["sarcasm_score"] = round(sarcasm_result["sarcasm_probability"] * 100, 2)
        sarcasm_result["sarcasm_label"] = "Likely Sarcastic"
        sarcasm_result["sarcasm_reason"] = "Positive wording follows negative context."
        return sarcasm_result

    # 8. Vague emotional wording
    if has_vague_uncertain_tone(text) and emotion in POSITIVE_EMOTIONS:
        sarcasm_result["sarcasm_probability"] = max(min(sarcasm_prob, 0.32), 0.30)
        sarcasm_result["sarcasm_score"] = round(sarcasm_result["sarcasm_probability"] * 100, 2)
        sarcasm_result["sarcasm_label"] = "Uncertain"
        sarcasm_result["sarcasm_reason"] = "Vague emotional wording."
        return sarcasm_result

    # 9. Default model decision
    if sarcasm_prob >= SARCASM_THRESHOLD:
        sarcasm_result["sarcasm_label"] = "Sarcastic"
    else:
        sarcasm_result["sarcasm_label"] = "Not Sarcastic"

    sarcasm_result["sarcasm_score"] = round(sarcasm_result["sarcasm_probability"] * 100, 2)
    sarcasm_result["sarcasm_reason"] = "Model probability used."

    return sarcasm_result

In [25]:
def adjust_emotion_with_sarcasm(text, emotion_result, sarcasm_result, previous_negative_context=False):
    emotion = emotion_result["primary_emotion"]
    emotion_score = emotion_result["emotion_score"]
    sarcasm_label = sarcasm_result["sarcasm_label"]
    sarcasm_prob = sarcasm_result["sarcasm_probability"]

    adjusted_emotion = emotion
    adjusted_score = emotion_score
    adjustment_reason = "No adjustment needed."

    contradiction = contradiction_strength(
        text,
        previous_negative_context=previous_negative_context,
    )

    is_sarcastic_like = sarcasm_label in {
        "Sarcastic",
        "Likely Sarcastic",
    }

    # 1. Neutral daily actions
    if is_literal_neutral_activity(text, emotion_result):
        adjusted_emotion = "neutral"
        adjusted_score = max(45.0, emotion_score * 0.60)
        adjustment_reason = "Neutral daily activity detected."
        return {
            "adjusted_emotion": adjusted_emotion,
            "adjusted_emotion_score": round(adjusted_score, 2),
            "adjustment_reason": adjustment_reason,
        }

        # 1.5 Supportive context should be positive/mixed-positive, not negative.
    if has_supportive_positive(text):
        adjusted_emotion = "gratitude"
        adjusted_score = max(60.0, emotion_score)
        adjustment_reason = "Supportive positive context detected."

        return {
            "adjusted_emotion": adjusted_emotion,
            "adjusted_emotion_score": round(adjusted_score, 2),
            "adjustment_reason": adjustment_reason,
        }

    # 1.6 Clear study preparation is positive.
    if has_clear_positive_study(text):
        adjusted_emotion = "optimism"
        adjusted_score = max(60.0, emotion_score)
        adjustment_reason = "Positive study/preparation context detected."

        return {
            "adjusted_emotion": adjusted_emotion,
            "adjusted_emotion_score": round(adjusted_score, 2),
            "adjustment_reason": adjustment_reason,
        }

    # 2. Positive recovery/completion
    if is_literal_positive_resolution(text):
        if emotion in NEGATIVE_EMOTIONS or emotion in NEUTRAL_EMOTIONS:
            adjusted_emotion = "joy"
            adjusted_score = max(55.0, emotion_score)
            adjustment_reason = "Positive resolution detected."
        else:
            adjusted_emotion = emotion
            adjusted_score = emotion_score
            adjustment_reason = "Positive resolution detected."

        return {
            "adjusted_emotion": adjusted_emotion,
            "adjusted_emotion_score": round(adjusted_score, 2),
            "adjustment_reason": adjustment_reason,
        }

    # 3. Stress / exhaustion
    if has_exhaustion_or_stress(text):
        adjusted_emotion = "sadness"
        adjusted_score = max(55.0, emotion_score)
        adjustment_reason = "Stress or exhaustion detected."

        return {
            "adjusted_emotion": adjusted_emotion,
            "adjusted_emotion_score": round(adjusted_score, 2),
            "adjustment_reason": adjustment_reason,
        }

    # 4. Vague positive prediction
    if sarcasm_label == "Uncertain" and emotion in POSITIVE_EMOTIONS:
        adjusted_emotion = "neutral"
        adjusted_score = max(40.0, emotion_score * 0.70)
        adjustment_reason = "Vague wording softened from positive to neutral."

        return {
            "adjusted_emotion": adjusted_emotion,
            "adjusted_emotion_score": round(adjusted_score, 2),
            "adjustment_reason": adjustment_reason,
        }

    # 5. Sarcasm over positive/neutral/surprise
    if is_sarcastic_like and sarcasm_prob >= SARCASM_THRESHOLD:
        if emotion in POSITIVE_EMOTIONS:
            adjusted_emotion = "annoyance"
            adjusted_score = max(60.0, emotion_score * 0.70)
            adjustment_reason = "Positive surface emotion adjusted due to sarcasm."

        elif emotion in {"neutral", "surprise", "confusion"} and (
            has_negative_event(text)
            or previous_negative_context
            or contradiction >= 2
        ):
            adjusted_emotion = "annoyance"
            adjusted_score = max(60.0, emotion_score * 0.75)
            adjustment_reason = "Neutral/uncertain emotion adjusted due to sarcastic negative context."

        elif emotion in NEGATIVE_EMOTIONS:
            adjusted_emotion = emotion
            adjusted_score = emotion_score
            adjustment_reason = "Emotion already negative; sarcasm kept as tone."

    # 6. Missed contradiction
    elif contradiction >= 2 and emotion in POSITIVE_EMOTIONS and has_negative_event(text):
        adjusted_emotion = "annoyance"
        adjusted_score = max(60.0, emotion_score * 0.70)

        sarcasm_result["sarcasm_label"] = "Likely Sarcastic"
        sarcasm_result["sarcasm_probability"] = max(
            sarcasm_result["sarcasm_probability"],
            0.70,
        )
        sarcasm_result["sarcasm_score"] = round(
            sarcasm_result["sarcasm_probability"] * 100,
            2,
        )

        adjustment_reason = "Positive emotion conflicts with negative context."

    return {
        "adjusted_emotion": adjusted_emotion,
        "adjusted_emotion_score": round(adjusted_score, 2),
        "adjustment_reason": adjustment_reason,
    }

In [26]:
def emotion_to_base_score(emotion):
    scores = {
        "joy": 80,
        "love": 75,
        "gratitude": 70,
        "optimism": 65,

        "neutral": 0,
        "confusion": -10,
        "surprise": 5,

        "annoyance": -55,
        "anger": -75,
        "sadness": -70,
        "fear": -65,
        "disappointment": -60,
    }

    return scores.get(emotion, 0)


def determine_sentiment(adjusted_emotion, sarcasm_result):
    sarcasm_label = sarcasm_result["sarcasm_label"]

    if sarcasm_label in {"Sarcastic", "Likely Sarcastic"}:
        return "Sarcastic / Negative"

    if sarcasm_label == "Uncertain":
        return "Uncertain / Mixed"

    if adjusted_emotion in POSITIVE_EMOTIONS:
        return "Positive"

    if adjusted_emotion in NEGATIVE_EMOTIONS:
        return "Negative"

    return "Neutral"


def calculate_mood_score(adjusted_emotion, emotion_score, sarcasm_result):
    sarcasm_label = sarcasm_result["sarcasm_label"]
    sarcasm_prob = sarcasm_result["sarcasm_probability"]

    base = emotion_to_base_score(adjusted_emotion)

    if (
        sarcasm_label in {"Sarcastic", "Likely Sarcastic"}
        and adjusted_emotion in {"annoyance", "surprise", "confusion", "neutral"}
    ):
        base = -60

    confidence = emotion_score / 100
    score = base * confidence

    if sarcasm_label in {"Sarcastic", "Likely Sarcastic"}:
        if score > 0:
            score = -abs(score) * 0.95
        else:
            score = score * (1 + min(sarcasm_prob, 0.65))

    elif sarcasm_label == "Uncertain":
        score = min(score, 5)

    return round(max(min(score, 100), -100), 2)

In [27]:
def analyze_statement(text, previous_negative_context=False):
    emotion_result = predict_emotion(text)
    sarcasm_result = predict_sarcasm(text)

    sarcasm_result = calibrate_sarcasm(
        text,
        emotion_result,
        sarcasm_result,
        previous_negative_context=previous_negative_context,
    )

    adjusted = adjust_emotion_with_sarcasm(
        text,
        emotion_result,
        sarcasm_result,
        previous_negative_context=previous_negative_context,
    )

    final_emotion = adjusted["adjusted_emotion"]
    final_emotion_score = adjusted["adjusted_emotion_score"]

    sentiment = determine_sentiment(
        final_emotion,
        sarcasm_result,
    )

    mood_score = calculate_mood_score(
        final_emotion,
        final_emotion_score,
        sarcasm_result,
    )

    interpretation = (
        f"The text appears {sentiment.lower()}. "
        f"Surface emotion was {emotion_result['primary_emotion']} "
        f"({emotion_result['emotion_score']}%). "
        f"Final interpreted emotion is {final_emotion} "
        f"({final_emotion_score}%). "
        f"{adjusted['adjustment_reason']} "
        f"Sarcasm reason: {sarcasm_result.get('sarcasm_reason', 'N/A')}"
    )

    return {
        "text": text,

        "surface_emotion": emotion_result["primary_emotion"],
        "surface_emotion_score": emotion_result["emotion_score"],
        "top3_emotions": emotion_result["top3_emotions"],

        "sarcasm_label": sarcasm_result["sarcasm_label"],
        "sarcasm_score": sarcasm_result["sarcasm_score"],
        "sarcasm_reason": sarcasm_result.get("sarcasm_reason", ""),

        "primary_emotion": final_emotion,
        "emotion_score": final_emotion_score,

        "sentiment": sentiment,
        "mood_score": mood_score,

        "interpretation": interpretation,
    }


def analyze_moodlens_text(text):
    statements = split_into_statements(text)

    statement_results = []
    previous_negative_context = False

    for statement in statements:
        result = analyze_statement(
            statement,
            previous_negative_context=previous_negative_context,
        )

        statement_results.append(result)

        if (
            result["primary_emotion"] in NEGATIVE_EMOTIONS
            or result["mood_score"] < -15
            or has_negative_event(statement)
        ):
            previous_negative_context = True
        else:
            previous_negative_context = False

    if not statement_results:
        return {
            "input_text": text,
            "statement_count": 0,
            "overall": {},
            "statements": [],
        }

    weighted_scores = []

    for item in statement_results:
        weight = 1.0

        if item["sarcasm_label"] in {"Sarcastic", "Likely Sarcastic"}:
            weight = 1.6

        if item["mood_score"] < -25:
            weight = max(weight, 1.4)

        weighted_scores.append((item["mood_score"], weight))

    total_weight = sum(weight for _, weight in weighted_scores)

    avg_score = round(
        sum(score * weight for score, weight in weighted_scores) / total_weight,
        2,
    )

    strong_sarcasm_count = sum(
        1
        for item in statement_results
        if item["sarcasm_label"] in {"Sarcastic", "Likely Sarcastic"}
    )

    uncertain_count = sum(
        1
        for item in statement_results
        if item["sarcasm_label"] == "Uncertain"
    )

    emotion_counts = {}

    for item in statement_results:
        emotion = item["primary_emotion"]
        emotion_counts[emotion] = emotion_counts.get(emotion, 0) + 1

    dominant_emotion = max(
        emotion_counts,
        key=emotion_counts.get,
    )

    if avg_score > 15:
        overall_mood = "Positive"
    elif avg_score < -15:
        overall_mood = "Negative"
    else:
        overall_mood = "Neutral"

    if strong_sarcasm_count > 0 and avg_score < -8:
        overall_mood = "Negative with Sarcasm"
    elif uncertain_count > 0 and overall_mood == "Neutral":
        overall_mood = "Neutral / Uncertain"
    elif strong_sarcasm_count > 0 and overall_mood == "Neutral":
        overall_mood = "Neutral / Mixed with Possible Sarcasm"

    trend = []

    for item in statement_results:
        if item["sarcasm_label"] in {"Sarcastic", "Likely Sarcastic"}:
            trend.append("sarcastic")
        elif item["sarcasm_label"] == "Uncertain":
            trend.append("uncertain")
        elif item["mood_score"] > 15:
            trend.append("positive")
        elif item["mood_score"] < -15:
            trend.append("negative")
        else:
            trend.append("neutral")

    return {
        "input_text": text,
        "statement_count": len(statement_results),
        "overall": {
            "overall_mood": overall_mood,
            "mood_score": avg_score,
            "dominant_emotion": dominant_emotion,
            "sarcasm_count": strong_sarcasm_count,
            "uncertain_count": uncertain_count,
            "trend": trend,
        },
        "statements": statement_results,
    }

In [28]:
def normalize_label(value):
    value = str(value).lower().strip()

    # Important: check "not sarcastic" before "sarcastic"
    if "not sarcastic" in value:
        return "not_sarcastic"

    if "negative with sarcasm" in value:
        return "negative_sarcasm"

    if "sarcastic" in value:
        return "sarcastic"

    if "uncertain" in value:
        return "uncertain"

    if "mixed" in value:
        return "mixed"

    if "positive" in value:
        return "positive"

    if "negative" in value:
        return "negative"

    if "neutral" in value:
        return "neutral"

    return value


def loose_match_prediction(expected_mood, predicted_mood):
    expected = normalize_label(expected_mood)
    predicted = normalize_label(predicted_mood)

    if expected == predicted:
        return True

    acceptable = {
        "negative_sarcasm": {
            "negative_sarcasm",
            "sarcastic",
            "negative",
            "mixed",
        },
        "sarcastic": {
            "sarcastic",
            "negative_sarcasm",
        },
        "positive": {
            "positive",
        },
        "negative": {
            "negative",
            "negative_sarcasm",
        },
        "neutral": {
            "neutral",
            "mixed",
            "uncertain",
        },
        "mixed": {
            "mixed",
            "neutral",
            "negative",
            "positive",
        },
        "uncertain": {
            "uncertain",
            "neutral",
            "mixed",
        },
    }

    return predicted in acceptable.get(expected, {expected})


def loose_match_sarcasm(expected_sarcasm, predicted_sarcasm):
    expected = normalize_label(expected_sarcasm)
    predicted = normalize_label(predicted_sarcasm)

    if expected == predicted:
        return True

    acceptable = {
        "sarcastic": {
            "sarcastic",
            "negative_sarcasm",
        },
        "not_sarcastic": {
            "not_sarcastic",
        },
        "uncertain": {
            "uncertain",
            "neutral",
            "mixed",
            "not_sarcastic",
        },
    }

    return predicted in acceptable.get(expected, {expected})


def get_predicted_sarcasm_from_result(result):
    labels = [
        item.get("sarcasm_label", "")
        for item in result.get("statements", [])
    ]

    if any(label in {"Sarcastic", "Likely Sarcastic"} for label in labels):
        return "Sarcastic"

    if any(label == "Uncertain" for label in labels):
        return "Uncertain"

    return "Not Sarcastic"

In [29]:
benchmark_examples = [
    # =========================================================
    # 1. Literal positive
    # Expected: Positive, not sarcastic
    # =========================================================
    {
        "text": "I finally completed my project and I feel proud.",
        "expected_mood": "Positive",
        "expected_sarcasm": "Not Sarcastic",
        "category": "literal_positive",
    },
    {
        "text": "Today was peaceful and I enjoyed spending time with my family.",
        "expected_mood": "Positive",
        "expected_sarcasm": "Not Sarcastic",
        "category": "literal_positive",
    },
    {
        "text": "I am grateful for the support I received today.",
        "expected_mood": "Positive",
        "expected_sarcasm": "Not Sarcastic",
        "category": "literal_positive",
    },
    {
        "text": "I solved a difficult bug and it made me really happy.",
        "expected_mood": "Positive",
        "expected_sarcasm": "Not Sarcastic",
        "category": "literal_positive",
    },
    {
        "text": "My presentation went well and I feel confident.",
        "expected_mood": "Positive",
        "expected_sarcasm": "Not Sarcastic",
        "category": "literal_positive",
    },

    # =========================================================
    # 2. Literal negative
    # Expected: Negative, not sarcastic
    # =========================================================
    {
        "text": "I feel exhausted and overwhelmed today.",
        "expected_mood": "Negative",
        "expected_sarcasm": "Not Sarcastic",
        "category": "literal_negative",
    },
    {
        "text": "The server crashed before the demo.",
        "expected_mood": "Negative",
        "expected_sarcasm": "Not Sarcastic",
        "category": "literal_negative",
    },
    {
        "text": "I failed the exam and I feel disappointed.",
        "expected_mood": "Negative",
        "expected_sarcasm": "Not Sarcastic",
        "category": "literal_negative",
    },
    {
        "text": "I am anxious about tomorrow's interview.",
        "expected_mood": "Negative",
        "expected_sarcasm": "Not Sarcastic",
        "category": "literal_negative",
    },
    {
        "text": "Everything went wrong today and I feel tired.",
        "expected_mood": "Negative",
        "expected_sarcasm": "Not Sarcastic",
        "category": "literal_negative",
    },

    # =========================================================
    # 3. Literal neutral
    # Expected: Neutral, not sarcastic
    # =========================================================
    {
        "text": "I went to college today and attended my lectures.",
        "expected_mood": "Neutral",
        "expected_sarcasm": "Not Sarcastic",
        "category": "literal_neutral",
    },
    {
        "text": "I had lunch and then studied for two hours.",
        "expected_mood": "Neutral",
        "expected_sarcasm": "Not Sarcastic",
        "category": "literal_neutral",
    },
    {
        "text": "The meeting started at 10 AM and ended at 11 AM.",
        "expected_mood": "Neutral",
        "expected_sarcasm": "Not Sarcastic",
        "category": "literal_neutral",
    },
    {
        "text": "I submitted the form and received a confirmation email.",
        "expected_mood": "Neutral",
        "expected_sarcasm": "Not Sarcastic",
        "category": "literal_neutral",
    },
    {
        "text": "I cleaned my room and arranged my books.",
        "expected_mood": "Neutral",
        "expected_sarcasm": "Not Sarcastic",
        "category": "literal_neutral",
    },

    # =========================================================
    # 4. Clear sarcasm
    # Positive surface + negative event
    # Expected: Negative with Sarcasm
    # =========================================================
    {
        "text": "Great, the app crashed during my final demo.",
        "expected_mood": "Negative with Sarcasm",
        "expected_sarcasm": "Sarcastic",
        "category": "clear_sarcasm",
    },
    {
        "text": "Amazing, my assignment got deleted right before submission.",
        "expected_mood": "Negative with Sarcasm",
        "expected_sarcasm": "Sarcastic",
        "category": "clear_sarcasm",
    },
    {
        "text": "Wonderful, another deadline appeared out of nowhere.",
        "expected_mood": "Negative with Sarcasm",
        "expected_sarcasm": "Sarcastic",
        "category": "clear_sarcasm",
    },
    {
        "text": "Perfect, my laptop stopped working before the exam.",
        "expected_mood": "Negative with Sarcasm",
        "expected_sarcasm": "Sarcastic",
        "category": "clear_sarcasm",
    },
    {
        "text": "Lovely, the code works except for the part where it doesn't.",
        "expected_mood": "Negative with Sarcasm",
        "expected_sarcasm": "Sarcastic",
        "category": "clear_sarcasm",
    },

    # =========================================================
    # 5. Emotional contradiction
    # Positive emotion word + negative social/event context
    # Expected: Negative with Sarcasm
    # =========================================================
    {
        "text": "I am so happy my best friend ignored me.",
        "expected_mood": "Negative with Sarcasm",
        "expected_sarcasm": "Sarcastic",
        "category": "emotional_contradiction",
    },
    {
        "text": "I really appreciate being blamed for something I did not do.",
        "expected_mood": "Negative with Sarcasm",
        "expected_sarcasm": "Sarcastic",
        "category": "emotional_contradiction",
    },
    {
        "text": "I love when people cancel plans after I get ready.",
        "expected_mood": "Negative with Sarcasm",
        "expected_sarcasm": "Sarcastic",
        "category": "emotional_contradiction",
    },
    {
        "text": "Thanks for making my already stressful day even harder.",
        "expected_mood": "Negative with Sarcasm",
        "expected_sarcasm": "Sarcastic",
        "category": "emotional_contradiction",
    },
    {
        "text": "I am grateful that my work was ignored again.",
        "expected_mood": "Negative with Sarcasm",
        "expected_sarcasm": "Sarcastic",
        "category": "emotional_contradiction",
    },

    # =========================================================
    # 6. Vague / uncertain
    # Expected: Neutral or uncertain, not strongly positive/negative
    # =========================================================
    {
        "text": "Today was certainly something.",
        "expected_mood": "Neutral",
        "expected_sarcasm": "Uncertain",
        "category": "vague_uncertain",
    },
    {
        "text": "That was an interesting experience.",
        "expected_mood": "Neutral",
        "expected_sarcasm": "Uncertain",
        "category": "vague_uncertain",
    },
    {
        "text": "I guess that happened.",
        "expected_mood": "Neutral",
        "expected_sarcasm": "Uncertain",
        "category": "vague_uncertain",
    },
    {
        "text": "Well, that was one way to spend the evening.",
        "expected_mood": "Neutral",
        "expected_sarcasm": "Uncertain",
        "category": "vague_uncertain",
    },
    {
        "text": "Okay then, that was unexpected.",
        "expected_mood": "Neutral",
        "expected_sarcasm": "Uncertain",
        "category": "vague_uncertain",
    },

    # =========================================================
    # 7. Coding / developer cases
    # =========================================================
    {
        "text": "The API finally works after three hours of debugging.",
        "expected_mood": "Positive",
        "expected_sarcasm": "Not Sarcastic",
        "category": "coding",
    },
    {
        "text": "Great, one small change broke the entire backend.",
        "expected_mood": "Negative with Sarcasm",
        "expected_sarcasm": "Sarcastic",
        "category": "coding",
    },
    {
        "text": "The model accuracy improved today and I feel motivated.",
        "expected_mood": "Positive",
        "expected_sarcasm": "Not Sarcastic",
        "category": "coding",
    },
    {
        "text": "Perfect, the deployment failed again after saying it was successful.",
        "expected_mood": "Negative with Sarcasm",
        "expected_sarcasm": "Sarcastic",
        "category": "coding",
    },
    {
        "text": "I don't understand why the same code works locally but fails online.",
        "expected_mood": "Negative",
        "expected_sarcasm": "Not Sarcastic",
        "category": "coding",
    },

    # =========================================================
    # 8. Student life cases
    # =========================================================
    {
        "text": "I studied well today and I feel prepared for the exam.",
        "expected_mood": "Positive",
        "expected_sarcasm": "Not Sarcastic",
        "category": "student_life",
    },
    {
        "text": "Wonderful, another surprise test before I even finished the last assignment.",
        "expected_mood": "Negative with Sarcasm",
        "expected_sarcasm": "Sarcastic",
        "category": "student_life",
    },
    {
        "text": "I am worried because I have too many deadlines this week.",
        "expected_mood": "Negative",
        "expected_sarcasm": "Not Sarcastic",
        "category": "student_life",
    },
    {
        "text": "Amazing, the exam got harder exactly when I thought I was ready.",
        "expected_mood": "Negative with Sarcasm",
        "expected_sarcasm": "Sarcastic",
        "category": "student_life",
    },
    {
        "text": "I completed my notes and felt relieved.",
        "expected_mood": "Positive",
        "expected_sarcasm": "Not Sarcastic",
        "category": "student_life",
    },

    # =========================================================
    # 9. Relationship / social context
    # =========================================================
    {
        "text": "My friend supported me when I was stressed.",
        "expected_mood": "Positive",
        "expected_sarcasm": "Not Sarcastic",
        "category": "social_context",
    },
    {
        "text": "I felt hurt because my friend ignored my message.",
        "expected_mood": "Negative",
        "expected_sarcasm": "Not Sarcastic",
        "category": "social_context",
    },
    {
        "text": "Lovely, everyone remembered me only when they needed help.",
        "expected_mood": "Negative with Sarcasm",
        "expected_sarcasm": "Sarcastic",
        "category": "social_context",
    },
    {
        "text": "Thanks for excluding me from the plan again.",
        "expected_mood": "Negative with Sarcasm",
        "expected_sarcasm": "Sarcastic",
        "category": "social_context",
    },
    {
        "text": "I had a calm conversation with my family today.",
        "expected_mood": "Positive",
        "expected_sarcasm": "Not Sarcastic",
        "category": "social_context",
    },

    # =========================================================
    # 10. Multi-line context
    # =========================================================
    {
        "text": """
        I woke up feeling okay.
        Then the assignment portal crashed before submission.
        Great, exactly what I needed before finals.
        """,
        "expected_mood": "Negative with Sarcasm",
        "expected_sarcasm": "Sarcastic",
        "category": "multiline_context",
    },
    {
        "text": """
        I prepared for the interview for two months.
        They rejected me in five minutes.
        Incredible experience.
        """,
        "expected_mood": "Negative with Sarcasm",
        "expected_sarcasm": "Sarcastic",
        "category": "multiline_context",
    },
    {
        "text": """
        I was nervous in the morning.
        But my presentation went well.
        Now I feel relieved and happy.
        """,
        "expected_mood": "Positive",
        "expected_sarcasm": "Not Sarcastic",
        "category": "multiline_context",
    },
    {
        "text": """
        The backend failed again.
        I fixed the issue after debugging for hours.
        I feel tired but satisfied.
        """,
        "expected_mood": "Mixed",
        "expected_sarcasm": "Not Sarcastic",
        "category": "multiline_context",
    },
    {
        "text": """
        Everything was going fine.
        Then my laptop froze.
        Perfect timing.
        """,
        "expected_mood": "Negative with Sarcasm",
        "expected_sarcasm": "Sarcastic",
        "category": "multiline_context",
    },

    # =========================================================
    # 11. Borderline short phrases
    # These are allowed to be uncertain/neutral
    # =========================================================
    {
        "text": "I am fine.",
        "expected_mood": "Neutral",
        "expected_sarcasm": "Uncertain",
        "category": "borderline",
    },
    {
        "text": "Sure, that makes sense.",
        "expected_mood": "Neutral",
        "expected_sarcasm": "Uncertain",
        "category": "borderline",
    },
    {
        "text": "Nice.",
        "expected_mood": "Neutral",
        "expected_sarcasm": "Uncertain",
        "category": "borderline",
    },
    {
        "text": "I guess I should be happy about this.",
        "expected_mood": "Neutral",
        "expected_sarcasm": "Uncertain",
        "category": "borderline",
    },
    {
        "text": "This is exactly what I expected.",
        "expected_mood": "Neutral",
        "expected_sarcasm": "Uncertain",
        "category": "borderline",
    },
]

In [30]:
def run_moodlens_benchmark(benchmark_examples):
    rows = []

    for i, item in enumerate(benchmark_examples, start=1):
        result = analyze_moodlens_text(item["text"])

        overall = result["overall"]

        predicted_mood = overall.get("overall_mood", "")
        predicted_sarcasm = get_predicted_sarcasm_from_result(result)

        mood_ok = loose_match_prediction(
            item["expected_mood"],
            predicted_mood,
        )

        sarcasm_ok = loose_match_sarcasm(
            item["expected_sarcasm"],
            predicted_sarcasm,
        )

        rows.append({
            "id": i,
            "category": item["category"],
            "text": " ".join(str(item["text"]).split()),
            "expected_mood": item["expected_mood"],
            "predicted_mood": predicted_mood,
            "mood_score": overall.get("mood_score", None),
            "expected_sarcasm": item["expected_sarcasm"],
            "predicted_sarcasm": predicted_sarcasm,
            "dominant_emotion": overall.get("dominant_emotion", None),
            "sarcasm_count": overall.get("sarcasm_count", 0),
            "uncertain_count": overall.get("uncertain_count", 0),
            "mood_ok": mood_ok,
            "sarcasm_ok": sarcasm_ok,
            "fully_ok": mood_ok and sarcasm_ok,
        })

    benchmark_df = pd.DataFrame(rows)

    print("\nOverall mood accuracy:")
    print(round(float(benchmark_df["mood_ok"].mean()), 4))

    print("\nSarcasm accuracy:")
    print(round(float(benchmark_df["sarcasm_ok"].mean()), 4))

    print("\nFully correct:")
    print(round(float(benchmark_df["fully_ok"].mean()), 4))

    print("\nCategory-wise:")
    display(
        benchmark_df.groupby("category")[["mood_ok", "sarcasm_ok", "fully_ok"]]
        .mean()
        .sort_values("fully_ok")
    )

    print("\nFailed cases:")
    display(
        benchmark_df[benchmark_df["fully_ok"] == False]
        .sort_values(["category", "id"])
    )

    return benchmark_df


benchmark_df = run_moodlens_benchmark(benchmark_examples)


Overall mood accuracy:
0.9091

Sarcasm accuracy:
0.9636

Fully correct:
0.9091

Category-wise:


,mood_ok,sarcasm_ok,fully_ok
category,,,
coding,0.8,1.0,0.8
multiline_context,0.8,1.0,0.8
literal_positive,0.8,1.0,0.8
social_context,0.8,0.8,0.8
student_life,0.8,0.8,0.8
clear_sarcasm,1.0,1.0,1.0
emotional_contradiction,1.0,1.0,1.0
borderline,1.0,1.0,1.0
literal_negative,1.0,1.0,1.0



Failed cases:


,id,category,text,expected_mood,predicted_mood,mood_score,expected_sarcasm,predicted_sarcasm,dominant_emotion,sarcasm_count,uncertain_count,mood_ok,sarcasm_ok,fully_ok
34,35,coding,I don't understand why the same code works loc...,Negative,Neutral,-9.33,Not Sarcastic,Not Sarcastic,confusion,0,0,False,True,False
2,3,literal_positive,I am grateful for the support I received today.,Positive,Neutral,0.00,Not Sarcastic,Not Sarcastic,neutral,0,0,False,True,False
47,48,multiline_context,I was nervous in the morning. But my presentat...,Positive,Neutral,1.27,Not Sarcastic,Not Sarcastic,fear,0,0,False,True,False
42,43,social_context,"Lovely, everyone remembered me only when they ...",Negative with Sarcasm,Positive,42.00,Sarcastic,Not Sarcastic,gratitude,0,0,False,False,False
36,37,student_life,"Wonderful, another surprise test before I even...",Negative with Sarcasm,Positive,44.00,Sarcastic,Not Sarcastic,joy,0,0,False,False,False


In [31]:
import json
import os

FUSION_SAVE_DIR = "/kaggle/working/moodlens_fusion_engine_v1"
os.makedirs(FUSION_SAVE_DIR, exist_ok=True)

fusion_config = {
    "version": "MoodLens Fusion Engine V1",
    "sarcasm_threshold": SARCASM_THRESHOLD,
    "max_length": MAX_LENGTH,

    "positive_emotions": sorted(list(POSITIVE_EMOTIONS)),
    "negative_emotions": sorted(list(NEGATIVE_EMOTIONS)),
    "neutral_emotions": sorted(list(NEUTRAL_EMOTIONS)),

    "positive_word_cues": POSITIVE_WORD_CUES,
    "negative_event_cues": NEGATIVE_EVENT_CUES,
    "contrast_cues": CONTRAST_CUES,
    "intensifier_cues": INTENSIFIER_CUES,
    "sarcasm_pattern_cues": SARCASM_PATTERN_CUES,
    "vague_uncertain_cues": VAGUE_UNCERTAIN_CUES,
    "positive_resolution_cues": POSITIVE_RESOLUTION_CUES,
    "neutral_activity_cues": NEUTRAL_ACTIVITY_CUES,

    "supportive_positive_cues": SUPPORTIVE_POSITIVE_CUES,
    "student_negative_cues": STUDENT_NEGATIVE_CUES,
    "coding_negative_cues": CODING_NEGATIVE_CUES,
    "clear_positive_study_cues": CLEAR_POSITIVE_STUDY_CUES,
    "expected_neutral_uncertain_cues": EXPECTED_NEUTRAL_UNCERTAIN_CUES,

    "emotion_score_map": {
        "joy": 80,
        "love": 75,
        "gratitude": 70,
        "optimism": 65,

        "neutral": 0,
        "confusion": -10,
        "surprise": 5,

        "annoyance": -55,
        "anger": -75,
        "sadness": -70,
        "fear": -65,
        "disappointment": -60,
    },

    "benchmark_result": {
        "mood_accuracy": 0.9091,
        "sarcasm_accuracy": 0.9636,
        "fully_correct": 0.9091,
    },
}

config_path = os.path.join(FUSION_SAVE_DIR, "fusion_config.json")

with open(config_path, "w") as f:
    json.dump(fusion_config, f, indent=4)

print("Saved fusion config:", config_path)

Saved fusion config: /kaggle/working/moodlens_fusion_engine_v1/fusion_config.json


In [32]:
benchmark_csv_path = os.path.join(FUSION_SAVE_DIR, "benchmark_results.csv")
benchmark_df.to_csv(benchmark_csv_path, index=False)

print("Saved benchmark results:", benchmark_csv_path)

Saved benchmark results: /kaggle/working/moodlens_fusion_engine_v1/benchmark_results.csv


In [33]:
readme = """
# MoodLens Fusion Engine V1

This is the production-style fusion layer for MoodLens.

It combines:
- Emotion model
- Sarcasm model
- Context-aware statement splitting
- General rule-based correction layer
- Multi-line mood aggregation

Current benchmark:
- Mood accuracy: 90.91%
- Sarcasm accuracy: 96.36%
- Fully correct: 90.91%

Architecture:
Text -> Split statements -> Emotion model -> Sarcasm model -> Fusion rules -> Final mood result

Important:
This is not a model-weight merge. It is an inference fusion engine.
"""

readme_path = os.path.join(FUSION_SAVE_DIR, "README.md")

with open(readme_path, "w") as f:
    f.write(readme)

print("Saved README:", readme_path)

Saved README: /kaggle/working/moodlens_fusion_engine_v1/README.md


In [34]:
import shutil

ZIP_PATH = "/kaggle/working/moodlens_fusion_engine_v1"

shutil.make_archive(
    ZIP_PATH,
    "zip",
    FUSION_SAVE_DIR,
)

print("Created ZIP:")
print(ZIP_PATH + ".zip")

Created ZIP:
/kaggle/working/moodlens_fusion_engine_v1.zip
